# GBDN submission H100 operator

This notebook is a thin orchestration interface. Scientific code, metrics, and training loops belong in tracked modules and subprocess runners. It is currently fail-closed: the final verification cell must fail until independent Gate-A acceptance and all frozen execution inputs and outputs exist. Legacy result trees are never writable from this workflow.

In [ ]:
import json, os, subprocess, sys
if 'torch' in sys.modules:
    raise RuntimeError('PyTorch was imported before GPU isolation')
from pathlib import Path
gpu_inventory = subprocess.run(['nvidia-smi', '-L'], check=True, capture_output=True, text=True).stdout
GPU_INDEX = os.environ.get('GBDN_H100_INDEX', '0')
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_INDEX
os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
if torch.cuda.device_count() != 1:
    raise RuntimeError(f'expected exactly one isolated GPU, found {torch.cuda.device_count()}')
gpu_name = torch.cuda.get_device_name(0)
if 'H100' not in gpu_name and os.environ.get('GBDN_ALLOW_NON_H100') != '1':
    raise RuntimeError(f'expected an NVIDIA H100, found {gpu_name}')
print(json.dumps({'physical_inventory': gpu_inventory.splitlines(), 'visible_device': gpu_name}, indent=2))

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / 'scripts' / 'run_submission.py').is_file():
    raise RuntimeError('run this notebook from the repository root')
OUTPUT_ROOT = ROOT / 'results_submission'
OFFICIAL_SPLITS = list(range(10))
TRAINING_SEEDS = [0, 1, 2]
MAX_WORKERS = 1
RERUN = False
CONTINUE_ON_ERROR = True
print(json.dumps({'root': str(ROOT), 'output_root': str(OUTPUT_ROOT), 'splits': OFFICIAL_SPLITS, 'training_seeds': TRAINING_SEEDS, 'max_workers': MAX_WORKERS}, indent=2))

## Execution phases

Phase launch cells will call reusable subprocess commands only after the verifier reports execution authorization. Until the independently reviewed acceptance token, confirmatory plan, baseline registry, and run plan are installed, no claim-bearing job is exposed here.

In [ ]:
preflight = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_submission.py'), 'verify', '--repository-root', str(ROOT)], capture_output=True, text=True, check=False)
preflight_report = json.loads(preflight.stdout)
if not preflight_report.get('ready_for_claim_bearing_execution', False):
    raise RuntimeError(f'CLAIM-BEARING EXECUTION BLOCKED: {preflight_report.get("execution_blockers", [])}')
launch = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_submission.py'), 'confirm', '--repository-root', str(ROOT), '--authoritative-dataset-root', str(ROOT / 'data')], text=True, check=False)
if launch.returncode != 0:
    raise RuntimeError(f'CONFIRMATORY SCHEDULER FAILED: exit={launch.returncode}')


In [ ]:
verification = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'run_submission.py'), 'verify', '--repository-root', str(ROOT)], capture_output=True, text=True, check=False)
print(verification.stdout)
if verification.returncode != 0:
    raise RuntimeError(f'SUBMISSION PIPELINE: FAIL (exit={verification.returncode})')
print('SUBMISSION PIPELINE: PASS')